# Instituto Tecnológico y de Estudios Superiores de Monterrey

## Proyecto: Particionamiento y Muestreo | PySpark

#### Steam Games Metadata and Player Reviews (2020–2024)
- Dataset link: [https://data.mendeley.com/datasets/jxy85cr3th/2]

**Equipo 29**

**Integrantes:**  

- Edmundo Carmona Galindo | A01796647
- Oliver Yousu Viveros Juarez | A01796912
- Martinez Ramirez Christian Gustavo | A01796999
- Diego Miguel Granados Gómex | A01337287

**Curso:** Análisis de Grandes Volúmenes de Datos

---

## Introducción

### Objetivo general

Construir una estrategia de particionamiento y muestreo representativo sobre el dataset Steam Games Metadata and Player Reviews (2020–2024), utilizando PySpark para procesar grandes volúmenes de datos y facilitar futuros análisis sobre comportamiento y engagement de usuarios.

## 1. Configuración del entorno

In [ ]:
pip install pyspark

In [1]:
import os
import json
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

os.environ["JAVA_HOME"] = "/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home"
os.environ["PATH"] = os.path.join(os.environ["JAVA_HOME"], "bin") + os.pathsep + os.environ["PATH"]

print(f"PySpark Ver: {pyspark.__version__}")

PySpark Ver: 4.1.1


In [2]:
# Creación de la SparkSession

spark = (
    SparkSession.builder
    .appName("ParticionamientoMuestreoSteam")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f"SparkSession Activa: {spark}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/12 19:34:04 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession Activa: <pyspark.sql.session.SparkSession object at 0x10cb6ef90>


## 2. Carga de datos

### 2.1 Definición de rutas de los archivos

In [10]:
RUTA_BASE = "/Users/edmundo.carmona/Personal/Master AI/Bigdata/Data/Steam Games Metadata and Player Reviews"

RUTA_GAMES_JSON = f"{RUTA_BASE}/games.jsonl"

### 2.2 Lectura del archivo metadata

In [11]:
games_df = spark.read.json(RUTA_GAMES_JSON)

print(f"Número de juegos: {games_df.count():,}")
print(f"Número de columnas: {len(games_df.columns)}")

Número de juegos: 65,686
Número de columnas: 22


### 2.3 Inspección del esquema

In [12]:
games_df.printSchema()

root
 |-- about_the_game: string (nullable = true)
 |-- app_id: string (nullable = true)
 |-- average_playtime_2weeks: long (nullable = true)
 |-- average_playtime_forever: long (nullable = true)
 |-- categories: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- detailed_description: string (nullable = true)
 |-- discount: string (nullable = true)
 |-- estimated_owners: string (nullable = true)
 |-- full_audio_languages: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- genres: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- median_playtime_2weeks: long (nullable = true)
 |-- median_playtime_forever: long (nullable = true)
 |-- name: string (nullable = true)
 |-- negative: long (nullable = true)
 |-- peak_ccu: long (nullable = true)
 |-- positive: long (nullable = true)
 |-- price: double (nullable = true)
 |-- release_date: string (nullable = true)
 |-- required_age: long (nullable = true)
 |-- short_de

## 3. Creación de variables de caracterización

Necesitamos crear tres variables derivadas:

1. **monetization**: Modelo de monetización (free-to-play vs. de pago)
2. **reception**: Ratio de recepción (positiva vs. mixta/negativa)
3. **period**: Periodo temporal (pandemia vs. post-pandemia)

### 3.1 Variable: monetization

In [13]:
games_df = games_df.withColumn(
    "monetization",
    F.when(F.col("price") == 0, "free")
     .otherwise("paid")
)

### 3.2 Variable: reception

In [14]:
games_df = games_df.withColumn(
    "reception",
    F.when(F.col("positive") / (F.col("positive") + F.col("negative")) >= 0.70, "positive")
    .otherwise("mixed_neg")
)


### 3.3 Variable: period

In [15]:
games_df = games_df.withColumn(
    "period",
    F.when(F.col("release_date").between("2020-01-01", "2021-12-31"), "pandemic")
    .when(F.col("release_date").between("2022-01-01", "2024-12-31"), "post_pandemic")
    .otherwise("unclassified")
)


### 3.4 Vista previa del DataFrame enriquecido

In [21]:

games_df.show(5)

+-------------------------------------+-------+-----------------------+------------------------+--------------------+-------------------------------------+--------+----------------+--------------------+--------------------+----------------------+-----------------------+--------------------------+--------+--------+--------+-----+------------+------------+-------------------------------------+--------------------+--------------------+------------+---------+------------+
|                       about_the_game| app_id|average_playtime_2weeks|average_playtime_forever|          categories|                 detailed_description|discount|estimated_owners|full_audio_languages|              genres|median_playtime_2weeks|median_playtime_forever|                      name|negative|peak_ccu|positive|price|release_date|required_age|                    short_description| supported_languages|                tags|monetization|reception|      period|
+-------------------------------------+-------+-------

## 4. Creación de particiones

Según el documento, debemos crear 10 particiones basadas en las combinaciones de:
- Monetización (2 valores: free, paid)
- Recepción (2 valores: positive, mixed_neg)
- Periodo (3 valores: pandemic, post_pandemic, unclassified)

Total de particiones posibles: 2 × 2 × 3 = 12 (se utilizarán 10)

### 4.1 Partición P1: Free-to-play + Positiva + Pandemia

In [ ]:
# TODO: Filtrar registros que cumplan:
# monetization = 'free' AND reception = 'positive' AND period = 'pandemic'

# P1 = ...
# print(f"Partición P1 - Count: {P1.count():,}")

### 4.2 Partición P2: Free-to-play + Positiva + Post-pandemia

In [ ]:
# TODO: Filtrar registros que cumplan:
# monetization = 'free' AND reception = 'positive' AND period = 'post_pandemic'


### 4.3 Partición P3: Free-to-play + Mixta/Negativa + Pandemia

In [ ]:
# TODO: Filtrar registros correspondientes a P3


### 4.4 Partición P4: Free-to-play + Mixta/Negativa + Post-pandemia

In [ ]:
# TODO: Filtrar registros correspondientes a P4


### 4.5 Partición P5: De pago + Positiva + Pandemia

In [ ]:
# TODO: Filtrar registros correspondientes a P5


### 4.6 Partición P6: De pago + Positiva + Post-pandemia

In [ ]:
# TODO: Filtrar registros correspondientes a P6


### 4.7 Partición P7: De pago + Mixta/Negativa + Pandemia

In [ ]:
# TODO: Filtrar registros correspondientes a P7


### 4.8 Partición P8: De pago + Mixta/Negativa + Post-pandemia

In [ ]:
# TODO: Filtrar registros correspondientes a P8


### 4.9 Particiones P9 y P10: Sin clasificar

In [ ]:
# TODO: P9 - Free-to-play + Positiva + Sin clasificar
# TODO: P10 - De pago + Positiva + Sin clasificar


## 5. Resumen de particiones

In [ ]:
# TODO: Crear tabla resumen con:
# - ID de partición
# - Descripción
# - Conteo de registros
# - Probabilidad observada


## 6. Muestreo estratificado

Se aplicará **muestreo aleatorio simple sin reemplazo (MASR)** dentro de cada partición.

### 6.1 Definición del tamaño de muestra

In [ ]:
# TODO: Definir tamaño total de muestra deseado
# SAMPLE_SIZE = ???


### 6.2 Muestreo proporcional por partición

In [ ]:
# TODO: Para cada partición:
# 1. Calcular fracción de muestreo = (tamaño_muestra_partición / tamaño_partición)
# 2. Aplicar sample(fraction=..., seed=42)


### 6.3 Combinación de muestras

In [ ]:
# TODO: Unir todas las submuestras en un único DataFrame
# muestra_final = P1_sample.union(P2_sample).union(...)


## 7. Validación de la muestra

### 7.1 Verificación de tamaños

In [ ]:
# TODO: Verificar que el tamaño de la muestra final sea cercano al objetivo


### 7.2 Comparación de distribuciones

In [ ]:
# TODO: Comparar distribución de variables en población vs. muestra
# Ejemplo: monetization, reception, period


## 8. Escritura de resultados

### 8.1 Guardado de particiones

In [ ]:
# TODO: Guardar cada partición en formato parquet
# P1.write.mode("overwrite").parquet("output/partitions/P1")


### 8.2 Guardado de la muestra final

In [ ]:
# TODO: Guardar muestra final
# muestra_final.write.mode("overwrite").parquet("output/muestra_estratificada")


In [ ]:
# Detener SparkSession
spark.stop()